#### Predictive Modelling for Multiphase Flow Regime Classification using High-Frequency Pressure Sensors

##### Part 2: Exploratory Data Analysis (EDA) and Model Training

##### **Purpose**: This notebook performs exploratory data analysis to visualize flow regimes and compares Linear/Logistic models against Random Forests for regime classification and pressure prediction.

##### **Author:** Bello Oluwatobi

##### **Last Updated:** November 24, 2025

### #1 Importing libraries and loading processed data

In [ ]:
# import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, confusion_matrix, mean_squared_error, r2_score
import joblib

In [ ]:
# loading the processed dataset 
df = pd.read_csv('../data/processed_data.csv')

### #2 Exploratory Data Analysis

In [ ]:
# plotting the flow regime map
plt.figure(figsize=(8, 6))

label_map = {0: 'Normal (Stable)', 1: 'Slugging (Unstable)'}

sns.scatterplot(
    data=df, 
    x='Vsg', 
    y='Vsl', 
    hue=df['Target_Label'].map(label_map), # Maps 0/1 to text
    palette={'Normal (Stable)': '#2ecc71', 'Slugging (Unstable)': '#e74c3c'}, # Green vs Red
    s=100,
    alpha=0.7
)

plt.title("Experimental Flow Regime Map: PRONTO Benchmark Dataset", fontsize=14)
plt.xlabel("Superficial Gas Velocity Vsg (m/s)", fontsize=12)
plt.ylabel("Superficial Liquid Velocity Vsl (m/s)", fontsize=12)

# Professional grid styling
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(title='Flow Regime', loc='upper right')

plt.savefig('../results/flow_regime_map.png', dpi=300)
plt.show()

In [ ]:
# plotting distribution plots for the pressure sensors


target_sensors = ['P4', 'p5', 'P6', 'P7', 'P8', 'P13', 'P14', 'P15', 'P17']

fig, axes = plt.subplots(3, 3, figsize=(18, 12))
axes = axes.flatten()

label_map = {0: 'Normal (Stable)', 1: 'Slugging (Unstable)'}

for i, sensor in enumerate(target_sensors):
    
    col_name = f'{sensor}_Mean'
    
    if col_name in df.columns:
        sns.kdeplot(data=df[df['Target_Label']==0][col_name], 
                    ax=axes[i], label='Normal', fill=True, color='#2ecc71')
        sns.kdeplot(data=df[df['Target_Label']==1][col_name], 
                    ax=axes[i], label='Slugging', fill=True, color='#e74c3c')
        
        axes[i].set_title(f"Distribution: {sensor}", fontsize=12)
        axes[i].set_xlabel("Mean Pressure (barg)")
        axes[i].set_ylabel("Density")
        axes[i].legend()

plt.tight_layout()
plt.suptitle("Pressure Distribution Comparison for the Sensor Set", fontsize=16, y=1.02)
plt.savefig('../results/pressure_sensors_distribution.png')
plt.show()

### #3 Data preparation

In [ ]:
# specifying features and target variables
feature_cols = ['Vsg', 'Vsl', 'Pipe_Diameter_m']

X = df[feature_cols]
y_class = df['Target_Label']


In [ ]:
# splitting the dataset into training and testing sets
X_train, X_test, y_class_train, y_class_test = train_test_split(
    X, y_class, test_size=0.2, random_state=42
)

In [ ]:
# scaling the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### #4 Model Training for Classification (Flow Regime)

In [ ]:
# training Logistic Regression model for classification
log_reg = LogisticRegression()
log_reg.fit(X_train_scaled, y_class_train)
y_pred_log = log_reg.predict(X_test_scaled)

In [ ]:
# training Random Forest Classifier for classification
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clf.fit(X_train_scaled, y_class_train)
y_pred_rf = rf_clf.predict(X_test_scaled)

In [ ]:
# comparing classification model performance
acc_log = accuracy_score(y_class_test, y_pred_log)
acc_rf = accuracy_score(y_class_test, y_pred_rf)

print(f"Logistic Regression Accuracy: {acc_log:.4f}")
print(f"Random Forest Accuracy: {acc_rf:.4f}")

### #5 Feature Importance Analysis

In [ ]:
# performing feature importance analysis on the superficial velocities
importances = rf_clf.feature_importances_
feature_names = X.columns

feat_import_df = pd.DataFrame({'Superficial Velocities': feature_names, 'Importance Score': importances})
feat_import_df = feat_import_df.sort_values(by='Importance Score', ascending=False)

plt.figure(figsize=(10, 4))
sns.barplot(data=feat_import_df, x='Importance Score', y='Superficial Velocities', palette='viridis')
plt.title("Operational Feature Importance (Liquid & Gas Superficial Velocities)", fontsize=14)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.savefig('../results/feature_importance_superficial_velocities.png', dpi=300)
plt.show()

In [ ]:
# performing feature importance analysis on the pressure sensors
sensor_cols = [f'{s}_Mean' for s in ['P4', 'p5', 'P6', 'P7', 'P8', 'P13', 'P14', 'P15', 'P17']]
X_sensor = df[sensor_cols]
y_sensor = df['Target_Label']

temp_rf = RandomForestClassifier().fit(X_sensor, y_sensor)
sens_importances = pd.DataFrame({'Pressure Sensor': sensor_cols, 'Importance Score': temp_rf.feature_importances_})
sens_importances = sens_importances.sort_values(by='Importance Score', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=sens_importances, x='Importance Score', y='Pressure Sensor', palette='viridis')
plt.title("Pressure Sensor Feature Importance", fontsize=14)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.savefig('../results/feature_importance_pressure_sensors.png', dpi=300)
plt.show()

### #6 Model Training for Regression (Pressure Sensors)

In [ ]:
# preparing sensor data for regression
y_p6_mean = df['P6_Mean'] # pressure at riser base
y_p8_mean = df['P8_Mean'] # pressure at middle of vertical riser

In [ ]:
# splitting the sensors p6 and p8 into training and testing sets
y_p6_mean_train, y_p6_mean_test, y_p8_mean_train, y_p8_mean_test = train_test_split(
    y_p6_mean, y_p8_mean, test_size=0.2, random_state=42
)

In [ ]:
# training linear regression and random forest regressor for P6
log_p6 = LinearRegression().fit(X_train_scaled, y_p6_mean_train)
reg_p6 = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train_scaled, y_p6_mean_train)

y_pred_log_reg_p6 = log_p6.predict(X_test_scaled)
rmse_log = np.sqrt(mean_squared_error(y_p6_mean_test, y_pred_log_reg_p6))
log_p6_score = r2_score(y_p6_mean_test, y_pred_log_reg_p6)

y_pred_rf_reg_p6 = reg_p6.predict(X_test_scaled)
rmse_rf = np.sqrt(mean_squared_error(y_p6_mean_test, y_pred_rf_reg_p6))
reg_p6_score = r2_score(y_p6_mean_test, y_pred_rf_reg_p6)

print(f"Linear Regression RMSE for P6: {rmse_log:.4f} barg")
print(f"P6 Pressure Prediction R² (Linear Regression): {log_p6_score:.3f}")
print("---"*20)
print(f"Random Forest Regressor RMSE for P6: {rmse_rf:.4f} barg")
print(f"P6 Pressure Prediction R² (Random Forest Regressor): {reg_p6_score:.3f}")

In [ ]:
# training regression models for P8 sensor
log_p8 = LinearRegression().fit(X_train_scaled, y_p8_mean_train)
reg_p8 = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train_scaled, y_p8_mean_train)

y_pred_log_reg_p8 = log_p8.predict(X_test_scaled)
rmse_log_p8 = np.sqrt(mean_squared_error(y_p8_mean_test, y_pred_log_reg_p8))
log_p8_score = r2_score(y_p8_mean_test, y_pred_log_reg_p8)

y_pred_rf_reg_p8 = reg_p8.predict(X_test_scaled)
rmse_rf_p8 = np.sqrt(mean_squared_error(y_p8_mean_test, y_pred_rf_reg_p8))
reg_p8_score = r2_score(y_p8_mean_test, y_pred_rf_reg_p8)

print(f"Linear Regression RMSE for P8: {rmse_log_p8:.4f} barg")
print(f"P8 Pressure Prediction R² (Linear Regression): {log_p8_score:.3f}")
print("---"*20)
print(f"Random Forest Regressor RMSE for P8: {rmse_rf_p8:.4f} barg")
print(f"P8 Pressure Prediction R² (Random Forest Regressor): {reg_p8_score:.3f}")

### #7 Saving Model Assets

In [ ]:
# saving the trained models and scaler
model_assets = {
    'classifier': rf_clf,
    'scaler': scaler,
    'reg_p6': reg_p6,
    'reg_p8': reg_p8
}

joblib.dump(model_assets, '../model_assets/models_assets.pkl')